# SceneAligner — Structured3D Runner

Unified Colab runner delegating directly to [`runners/pipeline.py`](runners/pipeline.py), matching [`runners/run_vm.py`](runners/run_vm.py) with persistent GCS storage (`gs://scenealigner` default).

- **After Run 1**: `.env`, `Structured3D_annotation_3d.zip`, prepared `s3d_data/<profile>/scenes_*.tar` slices, `test_scenes_3250_3500.tar`, and lossless `float32` eval `--recon_cache` shards are persisted in GCS so subsequent runs require **zero external Structured3D downloads**.
- **Queueing cells**: After running `gcp-auth` (if needed), `clone-install`, and `env-setup`, you can queue `prepare-s3d`, `train-s3d`, and `eval-s3d`.
- **Development**: Push local edits with `./runners/sync.sh` before running `clone-install`.

In [ ]:
# Authenticate to GCP on consumer Colab (no-op on Colab Enterprise or when gcloud is already active)
import shutil
import subprocess

_active_gcp = ""
if shutil.which("gcloud"):
    _active_gcp = subprocess.run(
        [
            "gcloud",
            "auth",
            "list",
            "--filter=status:ACTIVE",
            "--format=value(account)",
        ],
        capture_output=True,
        text=True,
    ).stdout.strip()
if not _active_gcp:
    try:
        from google.colab import auth

        auth.authenticate_user()
    except ImportError:
        pass

In [ ]:
import base64
import os
import subprocess
import sys

WORKSPACE = "/content/scenealigner_ws"
REPO_DIR = os.path.join(WORKSPACE, "Cs-5787-Project-Semantic-Localization")
REPO_URL = "https://github.com/lihengcao/Cs-5787-Project-Semantic-Localization.git"
BRANCH = "liheng"

os.makedirs(WORKSPACE, exist_ok=True)

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip()
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata

        GITHUB_TOKEN = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        GITHUB_TOKEN = ""

git_auth = []
if GITHUB_TOKEN:
    raw = f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
    b64 = base64.b64encode(raw).decode("ascii")
    git_auth = [
        "-c",
        f"http.https://github.com/.extraheader=Authorization: Basic {b64}",
    ]

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", *git_auth, "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", *git_auth, "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
        check=True,
    )

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!apt-get update -qq && apt-get install -y -qq p7zip-full aria2 libgl1

from runners import pipeline

pipeline.install_colab_deps()

In [ ]:
!nvidia-smi
!df -h /content

import os
import sys

WORKSPACE = "/content/scenealigner_ws"
REPO_DIR = os.path.join(WORKSPACE, "Cs-5787-Project-Semantic-Localization")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from runners import pipeline

# --- Run configuration ---
STAGES = ("prep", "train", "eval")
CHUNKS = [0, 1, 2]  # 0..14 (chunk 8 skipped automatically); ignored if ALL_TRAIN_CHUNKS=True
ALL_TRAIN_CHUNKS = False
SLICE_SCENES = 50

EPOCHS = 10
RESOLUTIONS = [512]
MIN_POINTS = 1000
SAVE_EVERY = 500
INIT_LORA = "jhcho99/SceneAligner/structured3d"
FREEZE_LORA = True
SEM_MODE = "consensus"
LAMBDA_SEM = 0.5
ALPHA_SEM = None
PHOTO_SEM_SOURCE = "gt"  # "gt" or "none"
INCLUDE_ROOMS = False
STRUCTURAL_PHOTO_SEMANTICS = False
USE_RECON_CACHE = True
EVAL_MODES = None
EVAL_CHECKPOINT_DIR = None

GCS_BACKUP_URI = None  # None -> .env or gs://scenealigner
NO_GCS = False
S3D_BASE_URL = None  # None -> .env / GCS config/.env (or interactive prompt only when external download needed)
RUN_ID = None  # None -> YYYY-MM-DD_HH-MM-SSZ_<slug> (auto-resumes matching incomplete ACTIVE_RUN)
NEW_RUN = False
TAG = None
KEEP_ZIPS = False
KEEP_RAW_VIEWS = False

SESSION = pipeline.init_session(
    workspace=WORKSPACE,
    stages=STAGES,
    chunks=CHUNKS,
    all_train_chunks=ALL_TRAIN_CHUNKS,
    slice_scenes=SLICE_SCENES,
    epochs=EPOCHS,
    resolutions=RESOLUTIONS,
    min_points=MIN_POINTS,
    save_every=SAVE_EVERY,
    init_lora=INIT_LORA,
    freeze_lora=FREEZE_LORA,
    sem_mode=SEM_MODE,
    lambda_sem=LAMBDA_SEM,
    alpha_sem=ALPHA_SEM,
    photo_sem_source=PHOTO_SEM_SOURCE,
    include_rooms=INCLUDE_ROOMS,
    structural_photo_semantics=STRUCTURAL_PHOTO_SEMANTICS,
    use_recon_cache=USE_RECON_CACHE,
    eval_modes=EVAL_MODES,
    eval_checkpoint_dir=EVAL_CHECKPOINT_DIR,
    gcs_uri=GCS_BACKUP_URI,
    no_gcs=NO_GCS,
    s3d_base_url=S3D_BASE_URL,
    run_id=RUN_ID,
    new_run=NEW_RUN,
    tag=TAG,
    keep_zips=KEEP_ZIPS,
    keep_raw_views=KEEP_RAW_VIEWS,
)

In [ ]:
# Optional smoke test: pytest + demo render
SESSION.run_smoke()

from IPython.display import Image, display

display(Image(f"{SESSION.paths['workspace']}/demo_S221/cameras.png", width=450))

In [ ]:
# Prepare Structured3D training slices (restores & heals GCS slices when available)
SESSION.run_prep()

In [ ]:
# Train Semantic Adapter / LoRA (strictly honors CHUNKS via --scene_ranges, auto-resumes & syncs checkpoints to GCS)
SESSION.run_train()

In [ ]:
# Evaluate on Structured3D test scenes (03250..03499) using GCS-backed test scenes & float32 recon cache, then record run summary
SESSION.run_eval()
SESSION.write_final_results()

In [ ]:
# Manual/full backup of prepared slices, test scenes, eval recon cache, and run outputs to GCS
SESSION.run_backup()